In [ ]:
!pip install transformers datasets torch

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving commands.csv to commands.csv


In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="commands.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 212
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 54
    })
})
{'text': 'go back a little', 'label': 1}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True
    )

dataset = dataset.map(tokenize)

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

In [ ]:
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    logging_steps=5,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,1.377242
10,1.395000
15,1.364199
20,1.293148
25,1.126637
30,0.933861
35,0.747549
40,0.548500
45,0.383269
50,0.231614


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (23 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=405, training_loss=0.12341381081460434, metrics={'train_runtime': 11974.2774, 'train_samples_per_second': 0.266, 'train_steps_per_second': 0.034, 'total_flos': 421261352386560.0, 'train_loss': 0.12341381081460434, 'epoch': 15.0})

In [ ]:
import torch

labels = ["FORWARD", "BACKWARD", "LEFT", "RIGHT"]

def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()
    return labels[pred]

# Test examples
print(predict("move forward"))
print(predict("venakki vellandi"))
print(predict("edama tirugu"))
print(predict("kudi tirugu"))

FORWARD
BACKWARD
LEFT
RIGHT


In [ ]:
model.save_pretrained("command_model")
tokenizer.save_pretrained("command_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('command_model/tokenizer_config.json', 'command_model/tokenizer.json')

In [ ]:
!zip -r command_model.zip command_model

  adding: command_model/ (stored 0%)
  adding: command_model/config.json (deflated 53%)
  adding: command_model/model.safetensors (deflated 8%)
  adding: command_model/tokenizer_config.json (deflated 42%)
  adding: command_model/tokenizer.json (deflated 71%)


In [ ]:
from google.colab import files
files.download("command_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch

labels = ["FORWARD", "BACKWARD", "LEFT", "RIGHT"]

def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()
    return labels[pred]

# 8 test cases
test_cases = [
    "move forward",
    "go ahead",
    "venakki vellandi",
    "reverse",
    "turn left",
    "edama tirugu",
    "turn right",
    "kudi tirugu"
]

print("TEST RESULTS:\n")

for i, text in enumerate(test_cases):
    result = predict(text)
    print(f"Test {i+1}: {text} → {result}")

TEST RESULTS:

Test 1: move forward → FORWARD
Test 2: go ahead → FORWARD
Test 3: venakki vellandi → BACKWARD
Test 4: reverse → BACKWARD
Test 5: turn left → LEFT
Test 6: edama tirugu → LEFT
Test 7: turn right → RIGHT
Test 8: kudi tirugu → RIGHT
